In [ ]:
from project_config import input_path, output_path, table_path, require_columns, BOOTSTRAP_RESAMPLES, RANDOM_SEED


In [ ]:
import numpy as np
import scipy.stats as stats
import matplotlib.pyplot as plt
from matplotlib import gridspec
import pandas as pd
import seaborn as sns

plt.rcParams.update({
    "font.family": "Arial",
    "font.size": 14,
    "axes.linewidth": 1.2,
    "axes.titlesize": 14,
    "axes.labelsize": 14,
    "xtick.labelsize": 12,
    "ytick.labelsize": 12,
    'grid.linestyle':    '--',
    'grid.linewidth':    0.4,
    'axes.linewidth':   0.8,
})

def clean_axis(ax, xlabel="", ylabel=""):
    """Apply minimal Nature-style axis formatting."""
    ax.set_xlabel(xlabel)
    ax.set_ylabel(ylabel)

    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)

    ax.tick_params(direction="out", length=4, width=1.1)

# Load data

In [ ]:
df1 = pd.read_parquet(input_path('data_first_au.parquet'))
df2 = pd.read_parquet(input_path('data_last_au.parquet'))
require_columns(df1, ["AIProb"])
if not df1["AIProb"].dropna().between(0, 1).all():
    raise ValueError("AIProb must be a probability in [0, 1]; convert percentages before loading.")

require_columns(df2, ["AIProb"])
if not df2["AIProb"].dropna().between(0, 1).all():
    raise ValueError("AIProb must be a probability in [0, 1]; convert percentages before loading.")


In [ ]:
country_gap_first = (
    df1
    .dropna(
        subset=[
            "first_country",
            "first_Female",
            "AIProb"
        ]
    )
    .groupby(
        ["first_country", "first_Female"]
    )["AIProb"]
    .agg(
        mean="mean",
        n="size"
    )
    .reset_index()
)

country_gap_first = (
    country_gap_first
    .pivot(
        index="first_country",
        columns="first_Female"
    )
)

country_gap_first = country_gap_first.reindex(columns=pd.MultiIndex.from_product([["mean", "n"], [0.0, 1.0]]))

country_gap_first.columns = [
    "male_AIProb",
    "female_AIProb",
    "male_n",
    "female_n"
]

country_gap_first = (
    country_gap_first
    .reset_index()
)

country_gap_first["gap"] = (
    country_gap_first["female_AIProb"]
    -
    country_gap_first["male_AIProb"]
)

country_gap_first = (
    country_gap_first
    .sort_values(
        "gap",
        ascending=False
    )
)

In [ ]:
country_gap_last = (
    df2
    .dropna(
        subset=[
            "last_country",
            "last_Female",
            "AIProb"
        ]
    )
    .groupby(
        ["last_country", "last_Female"]
    )["AIProb"]
    .agg(
        mean="mean",
        n="size"
    )
    .reset_index()
)

country_gap_last = (
    country_gap_last
    .pivot(
        index="last_country",
        columns="last_Female"
    )
)

country_gap_last = country_gap_last.reindex(columns=pd.MultiIndex.from_product([["mean", "n"], [0.0, 1.0]]))

country_gap_last.columns = [
    "male_AIProb",
    "female_AIProb",
    "male_n",
    "female_n"
]

country_gap_last = (
    country_gap_last
    .reset_index()
)

country_gap_last["gap"] = (
    country_gap_last["female_AIProb"]
    -
    country_gap_last["male_AIProb"]
)

country_gap_last = (
    country_gap_last
    .sort_values(
        "gap",
        ascending=False
    )
)

In [ ]:
country_gap_first = country_gap_first.rename(
    columns={
        "first_country": "country"
    }
)

country_gap_last = country_gap_last.rename(
    columns={
        "last_country": "country"
    }
)

In [ ]:
import geopandas as gpd
import matplotlib.pyplot as plt
import matplotlib as mpl
import cartopy.crs as ccrs
import cartopy.feature as cfeature
import country_converter as coco


# -----------------------
# Load world map
# -----------------------

shapefile = input_path('ne_10m_admin_0_countries.shp')

world_base = gpd.read_file(shapefile)

world_base["country"] = coco.convert(
    names=world_base["SOVEREIGNT"],
    to="ISO2",
    not_found=None
)


# -----------------------
# Prepare country tables
# -----------------------
# Assumes country_gap_first / country_gap_last have:
# country, male_AIProb, female_AIProb, gap

country_gap_first = country_gap_first.copy()
country_gap_last = country_gap_last.copy()

for d in [country_gap_first, country_gap_last]:
    d["male_AIProb_pct"] = d["male_AIProb"] * 100
    d["female_AIProb_pct"] = d["female_AIProb"] * 100
    d["gap_pct"] = d["gap"] * 100


# -----------------------
# Merge with map
# -----------------------

world_first = world_base.merge(
    country_gap_first,
    on="country",
    how="left"
)

world_last = world_base.merge(
    country_gap_last,
    on="country",
    how="left"
)


# -----------------------
# Color scales
# -----------------------

seq_cols = [
    "male_AIProb_pct",
    "female_AIProb_pct"
]

seq_values = pd.concat([
    world_first[seq_cols].stack(),
    world_last[seq_cols].stack()
]).dropna()
seq_vmax = float(seq_values.quantile(0.98)) if not seq_values.empty else 0.0
if not np.isfinite(seq_vmax) or seq_vmax <= 0:
    seq_vmax = 1.0

seq_norm = mpl.colors.Normalize(
    vmin=0,
    vmax=seq_vmax
)

gap_values = pd.concat([
    world_first["gap_pct"].abs(),
    world_last["gap_pct"].abs()
]).dropna()
gap_abs = float(gap_values.quantile(0.98)) if not gap_values.empty else 0.0
if not np.isfinite(gap_abs) or gap_abs <= 0:
    gap_abs = 1.0

gap_norm = mpl.colors.TwoSlopeNorm(
    vmin=-gap_abs,
    vcenter=0,
    vmax=gap_abs
)

seq_cmap = mpl.cm.YlOrRd
gap_cmap = mpl.cm.RdBu_r


# -----------------------
# Plot function
# -----------------------

def plot_world_panel(ax, world, col, title, cmap, norm):
    ax.set_extent(
        [-150, 180, -55, 70],
        crs=ccrs.PlateCarree()
    )

    ax.set_facecolor("white")

    ax.add_feature(
        cfeature.OCEAN,
        color="white",
        zorder=0
    )

    ax.add_feature(
        cfeature.LAND,
        color="lightgrey",
        zorder=0
    )

    world.plot(
        ax=ax,
        transform=ccrs.PlateCarree(),
        column=col,
        cmap=cmap,
        norm=norm,
        missing_kwds={
            "color": "lightgrey"
        },
        edgecolor="black",
        linewidth=0.12,
        zorder=2
    )

    ax.set_title(
        title,
        pad=8
    )

    ax.axis("off")

In [ ]:
# -----------------------
# Figure: 3 rows × 2 cols
# -----------------------

fig = plt.figure(figsize=(13, 5.5))

gs = fig.add_gridspec(
    2,
    3,
    wspace=0.03,
    hspace=0.04
)

axes = []

for i in range(2):
    for j in range(3):
        ax = fig.add_subplot(
            gs[i, j],
            projection=ccrs.Robinson()
        )
        axes.append(ax)


# Row 1: First author
plot_world_panel(
    axes[0],
    world_first,
    "male_AIProb_pct",
    "First author: male-led AI probability",
    seq_cmap,
    seq_norm
)

plot_world_panel(
    axes[1],
    world_first,
    "female_AIProb_pct",
    "First author: female-led AI probability",
    seq_cmap,
    seq_norm
)

plot_world_panel(
    axes[2],
    world_first,
    "gap_pct",
    "First author: female − male gap",
    gap_cmap,
    gap_norm
)


# Row 2: Last author
plot_world_panel(
    axes[3],
    world_last,
    "male_AIProb_pct",
    "Last author: male-led AI probability",
    seq_cmap,
    seq_norm
)

plot_world_panel(
    axes[4],
    world_last,
    "female_AIProb_pct",
    "Last author: female-led AI probability",
    seq_cmap,
    seq_norm
)

plot_world_panel(
    axes[5],
    world_last,
    "gap_pct",
    "Last author: female − male gap",
    gap_cmap,
    gap_norm
)



# -----------------------
# Colorbars
# -----------------------

cax1 = fig.add_axes([0.18, 0.055, 0.30, 0.015])
cb1 = mpl.colorbar.ColorbarBase(
    cax1,
    cmap=seq_cmap,
    norm=seq_norm,
    orientation="horizontal"
)

cb1.set_label(
    "Mean AI probability (%)"
)

cb1.ax.tick_params()


cax2 = fig.add_axes([0.55, 0.055, 0.30, 0.015])
cb2 = mpl.colorbar.ColorbarBase(
    cax2,
    cmap=gap_cmap,
    norm=gap_norm,
    orientation="horizontal"
)

cb2.set_label(
    "Female − male gap in AI probability (percentage points)"
)



# -----------------------
# Panel labels
# -----------------------

for ax, label in zip(axes, list("ABCDEF")):
    ax.text(
        0.01,
        0.98,
        label,
        transform=ax.transAxes,
        fontweight="bold",
        va="top",
        ha="left"
    )


plt.subplots_adjust(
    left=0.02,
    right=0.98,
    top=0.95,
    bottom=0.10
)

plt.savefig(output_path('Fig_4_country/fig3_country.pdf'), bbox_inches="tight")